In [1]:
import os
os.getcwd()

from pathlib import Path
from stgae.config.load_config import load_config
from stgae.data.preproccesing import get_columns
import pandas as pd

paths = load_config()['paths']    
data_path = Path(paths['raw_data'])

In [ ]:
output_path = data_path
output_file_name = 'filtered_data.csv'
columns = get_columns()

output_file = output_path / output_file_name

#fOLD, but keep to do tests
filters = lambda df: (
    (df["date"] == '2004-03-01') &
    (df["time"] < '04:00:00')
)

chunk_size = 100_000

first = True
for chunk in pd.read_csv(data_path / 'data.txt', sep=' ', names= columns, chunksize=chunk_size):
    subset = chunk[filters(chunk)]
    if not subset.empty:
        subset.to_csv(
            output_file,
            mode="w" if first else "a",
            index=False,
            header=first,
        )
        first = False

In [ ]:
filtered_df = pd.read_csv(output_file)
filtered_df.head()

,date,time,epoch,moteid,temperature,humidity,light,voltage
0,2004-03-01,00:01:57.13085,5648,1,18.4498,43.1191,43.24,2.67532
1,2004-03-01,00:02:50.458234,5650,1,18.4400,43.0858,43.24,2.66332
2,2004-03-01,00:04:26.606602,5653,1,18.4400,43.1191,43.24,2.65143
3,2004-03-01,00:05:28.379208,5655,1,18.4498,43.0524,43.24,2.65143
4,2004-03-01,00:05:50.456126,5656,1,18.4302,43.1525,43.24,2.66332


Let's build the graph.
I decided to use a graph using knn (with k=5), and the edges weighted according to the inverse of the distance. 

Building adjacency matrix

In [ ]:
import numpy as np
from stgae.data.preproccesing import calculate_distances, calculate_adjacency_matrix

coordinates = pd.read_csv(paths['data_root'] / 'sensor_coordinates.txt', sep=' ')
dist_matrix = calculate_distances(coordinates) #numpy array

k=5 #for k-nearest neighbors adj matrix

A = calculate_adjacency_matrix(dist_matrix, k=k) #torch tensor

Now implement thw windowed dataset